# 04 - Attentive FP (State-of-the-Art GNN)
## Predicting Human Intestinal Absorption (HIA)
**Student:** Mohammad Karamath Fardeen | ID: 25251265  
**Supervisor:** Kolawole Adebayo | Maynooth University | 2025-2026

Model: Attentive FP (Xiong et al., J. Med. Chem. 2020)  
Uses graph attention mechanisms at both atom and molecule level

In [ ]:
import os, numpy as np, pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import AttentiveFP
from rdkit import Chem
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, matthews_corrcoef

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Reuse graph construction from notebook 03
ATOM_LIST = ['C','N','O','S','F','Cl','Br','I','P','B','Si','H','Other']

def atom_features(atom):
    symbol = atom.GetSymbol()
    enc = [1 if symbol == s else 0 for s in ATOM_LIST[:-1]]
    enc.append(1 if symbol not in ATOM_LIST[:-1] else 0)
    return enc + [atom.GetDegree(), atom.GetFormalCharge(),
                  int(atom.GetHybridization()), int(atom.GetIsAromatic()),
                  atom.GetTotalNumHs()]

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.rdchem.BondType.SINGLE),
            int(bt == Chem.rdchem.BondType.DOUBLE),
            int(bt == Chem.rdchem.BondType.TRIPLE),
            int(bt == Chem.rdchem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]

def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_indices, edge_feats = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_features(bond)
        edge_indices += [[i,j],[j,i]]
        edge_feats   += [feat, feat]
    if not edge_indices:
        edge_index = torch.zeros((2,0), dtype=torch.long)
        edge_attr  = torch.zeros((0,6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_feats, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float))

class HIAGraphDataset(InMemoryDataset):
    def __init__(self, csv_path):
        super().__init__(None)
        df = pd.read_csv(csv_path)
        data_list = [g for g in [smiles_to_graph(r['Drug'], int(r['Y']))
                                  for _, r in df.iterrows()] if g is not None]
        print(f'Loaded {len(data_list)} graphs from {csv_path}')
        self.data, self.slices = self.collate(data_list)

print('Setup complete!')

## 1. Load Data

In [ ]:
train_ds = HIAGraphDataset('data/raw/hia_train.csv')
valid_ds = HIAGraphDataset('data/raw/hia_valid.csv')
test_ds  = HIAGraphDataset('data/raw/hia_test.csv')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

num_node_features = train_ds[0].x.shape[1]
num_edge_features = train_ds[0].edge_attr.shape[1]
print(f'Node features: {num_node_features} | Edge features: {num_edge_features}')

## 2. Build and Train Attentive FP

In [ ]:
# Attentive FP from PyTorch Geometric
model = AttentiveFP(
    in_channels=num_node_features,
    hidden_channels=64,
    out_channels=1,
    edge_dim=num_edge_features,
    num_layers=3,
    num_timesteps=2,
    dropout=0.2,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

labels = [int(d.y.item()) for d in train_ds]
n_pos, n_neg = sum(labels), len(labels) - sum(labels)
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            probs = torch.sigmoid(out.squeeze(-1))
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend((probs > 0.5).float().cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())
    return {
        'roc_auc':  round(roc_auc_score(all_labels, all_probs), 4),
        'f1':       round(f1_score(all_labels, all_preds), 4),
        'accuracy': round(accuracy_score(all_labels, all_preds), 4),
        'mcc':      round(matthews_corrcoef(all_labels, all_preds), 4),
    }

best_val_auc, best_state, patience_counter = 0.0, None, 0

print('Training Attentive FP...')
for epoch in range(1, 151):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = criterion(out.squeeze(-1), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    val_metrics = evaluate(model, valid_loader)
    scheduler.step(1 - val_metrics['roc_auc'])

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Loss: {total_loss/len(train_ds):.4f} | Val AUC: {val_metrics["roc_auc"]:.4f}')

    if val_metrics['roc_auc'] > best_val_auc:
        best_val_auc = val_metrics['roc_auc']
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 20:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
torch.save(model.state_dict(), 'results/metrics/attentivefp_model.pt')

test_metrics = evaluate(model, test_loader)
print(f'\nAttentive FP Test Results:')
for k, v in test_metrics.items():
    print(f'  {k}: {v}')